In [ ]:
import numpy as np
import utils
import make_dag
from os.path import exists
import json
import glob
import os

In [ ]:
logdm2_list = np.asarray(np.ndarray.tolist(np.arange(-2.10, 2.10, 0.1))+ [2.15])
dm2_list = np.power(10, logdm2_list)  
dm2_list = dm2_list[np.logical_and(dm2_list>.1, dm2_list<50)]
dm2_list = np.append(dm2_list, 50.0)

log_Um4sq_list = np.asarray(np.ndarray.tolist(np.arange(-3., 0., 0.1)))  
Um4sq_list = np.power(10, log_Um4sq_list)
Um4sq_list = Um4sq_list[Um4sq_list<0.50]
Um4sq_list = np.append(Um4sq_list, 0.50)

log_Ut4sq_list = np.asarray(np.ndarray.tolist(np.arange(-3., 0., 0.2)))   
Ut4sq_list = np.power(10, log_Ut4sq_list)  
Ut4sq_list = np.insert(Ut4sq_list, 0, 0.)
Ut4sq_list = Ut4sq_list[Ut4sq_list<0.50]
Ut4sq_list = np.append(Ut4sq_list, 0.50)

d24_list = [np.pi]

In [ ]:
def doesfilehaveBayes(filename, delete = False):
    
    exist = exists(filename)
    
    if exist:
        with open(filename) as f:
            data = json.load(f)
            hasbayes = "BayesEvidence" in data
            
            if hasbayes == False:
                print(filename, "does not have BayesEvidence")
                
                if delete:
                    print("Deleting")
                    filebase = filename.replace('.json', '')
                    filenames = glob.glob(filebase+"*")

                    for filetodelete in filenames:

                        os.remove(filetodelete)
            return hasbayes
    
    else:
        return False

In [ ]:
FS = '2.00'
crSlopeWidth = '0.25'


fit_dict = {
    'use_fastMC' : 'True',
    'Emax' : '9976',
    'sim' : 'Platinum_Split_Plus_Tau',
    'max_sim' : '-1', 
    'fast_scaling' : FS,
    'fastmc_dir' : '/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/FastMC/',
    'realization_file' : "/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/scripts/jobs/Data/Data_IC86.All_cut_everything_platinum_LT_240747841.npz",
    'n_seeds' : '5',
    'years' : '7.634',
    'osg' : 'False',
    'bayes' : 'True',
    'crSlopeWidth' : crSlopeWidth
}

def print_vars(f, jobname, var_name, val):
    f.write('VARS\t' + jobname.strip() + '\t' + var_name+'=\"' + val+ "\"\n")
    
def make_fit_dag(filename, jobname, fluxdir, point, outdir, osg = "True"):
    if osg == "True":
        filename.write('JOB\t' + jobname + '\t/home/adiaz/IC-MEOWS-th34/FitsToData/Bayes/do_fit_OSG.submit\n')
    else:
        filename.write('JOB\t' + jobname + '\t/home/adiaz/IC-MEOWS-th34/FitsToData/Bayes/do_fit.submit\n')
    print_vars(filename, jobname, 'point', point)
    print_vars(filename, jobname, 'fluxdir', fluxdir)
    print_vars(filename, jobname, 'outdir', outdir)
    for key, value in fit_dict.items():
        print_vars(filename, jobname, key, value)

In [ ]:
dag_filename = './evidence_tolerance_0.5_n_live_points_300_sampling_efficiency_0.5_crSlopeWidth_{crSlopeWidth}.dag'.format(
    crSlopeWidth = crSlopeWidth)
dirname = '/data/user/adiaz/Bayes_crSlopeWidth_0.25/'


osg = fit_dict['osg']
Ue4sq= 0.
del14=0.
with open(dag_filename, 'w') as f:
    i=0
    for i_ut4sq, Ut4sq in enumerate(Ut4sq_list):

        print(Ut4sq)
        if Ut4sq == 0.:

            
            prefix =  "{:.6f}_{:.6f}_{:.6f}.json".format(0.,0.,0.,)
            point = utils.get_point_string()
            fluxdir = "/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/Fluxes/AIRS_mceq121_2011_sib_HG_E3/Ut4sq_" + str(format(float(Ut4sq), '.6f')) + '/'
            
            outdir = dirname + "/Ut4sq_{:.6f}/".format(Ut4sq)
            outfilename = outdir + prefix
            
            if not doesfilehaveBayes(outfilename):

                make_fit_dag(f, str(i), fluxdir, point, dirname + "/Ut4sq_{:.6f}/".format(Ut4sq), osg)
                print_vars(f, str(i), 'prefix', prefix.replace('.json', ''))

                i+=1
        for i_um4sq, Um4sq in enumerate(Um4sq_list):
            for i_dm2, dm2 in enumerate(dm2_list):
                for del24 in d24_list:

                    th14, th24, th34 = utils.Usq_to_thetas(Ue4sq=Ue4sq, Um4sq=Um4sq, Ut4sq=Ut4sq)
                    point = utils.get_point_string(dm2 = dm2, th14 = th14, th24 = th24, th34 = th34, d24 = np.pi)
                    prefix =  "{:.6f}_{:.6f}_{:.6f}.json".format(dm2, th24, th34)
            
                    fluxdir = "/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/Fluxes/AIRS_mceq121_2011_sib_HG_E3/Ut4sq_" + str(format(float(Ut4sq), '.6f')) + '/'
                    outdir =  dirname + "/Ut4sq_{:.6f}/".format(Ut4sq)
                    outfilename = outdir + prefix

                    if not doesfilehaveBayes(outfilename, False):

                        print(Um4sq_list[i_um4sq], Ut4sq_list[i_ut4sq])
                        print(utils.Usq_to_thetas(0.,Um4sq = Um4sq_list[i_um4sq], Ut4sq=Ut4sq_list[i_ut4sq]))
                        make_fit_dag(f, str(i), fluxdir, point, outdir, osg)
                        print_vars(f, str(i), 'prefix', prefix.replace('.json', ''))

                        i+=1


In [ ]:
# loose = np.load("bayes_evidence_tolerance_0.5_n_live_points_300_sampling_efficiency_0.5_Run2.npy")
# loose_null = np.load("bayes_evidence_tolerance_0.5_n_live_points_300_sampling_efficiency_0.5_Run2_null.npy")

In [ ]:
dag_filename = './evidence_tolerance_0.5_n_live_points_300_sampling_efficiency_0.5_crSlopeWidth_{crSlopeWidth}.dag'.format(
    crSlopeWidth = crSlopeWidth)
dirname = '/data/user/adiaz/Bayes_crSlopeWidth_0.25/'

# dag_filename ='./evidence_tolerance_0.5_n_live_points_300_sampling_efficiency_0.5.dag'
# dag_filename ='./0to1.dag'
# dag_filename = 'test.dag'

# dirname = "/data/user/adiaz/Bayes_Run3/"
# dirname = "/data/user/adiaz/Bayes_Run2/"
# dirname = "/data/user/adiaz/Bayes/"

osg = fit_dict['osg']
Ue4sq= 0.
del14=0.
with open(dag_filename, 'w') as f:
    i=0
    for i_ut4sq, Ut4sq in enumerate(Ut4sq_list):

        print(Ut4sq)
        if Ut4sq == 0.:

            
            prefix =  "{:.6f}_{:.6f}_{:.6f}.json".format(0.,0.,0.,)
            point = utils.get_point_string()
            fluxdir = "/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/Fluxes/AIRS_mceq121_2011_sib_HG_E3/Ut4sq_" + str(format(float(Ut4sq), '.6f')) + '/'
            
            outdir = dirname + "/Ut4sq_{:.6f}/".format(Ut4sq)
            outfilename = outdir + prefix
            
            if not doesfilehaveBayes(outfilename):

                make_fit_dag(f, str(i), fluxdir, point, dirname + "/Ut4sq_{:.6f}/".format(Ut4sq), osg)
                print_vars(f, str(i), 'prefix', prefix.replace('.json', ''))
#                 f.write('Retry ' + str(i) + " 10\n")

                i+=1
        for i_um4sq, Um4sq in enumerate(Um4sq_list):
#                 print(Um4sq)
            for i_dm2, dm2 in enumerate(dm2_list):
#                     print(dm2)
                for del24 in d24_list:

                    th14, th24, th34 = utils.Usq_to_thetas(Ue4sq=Ue4sq, Um4sq=Um4sq, Ut4sq=Ut4sq)
                    point = utils.get_point_string(dm2 = dm2, th14 = th14, th24 = th24, th34 = th34, d24 = np.pi)
                    prefix =  "{:.6f}_{:.6f}_{:.6f}.json".format(dm2, th24, th34)
#                         print(prefix)
            
                    fluxdir = "/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/Fluxes/AIRS_mceq121_2011_sib_HG_E3/Ut4sq_" + str(format(float(Ut4sq), '.6f')) + '/'
                    outdir =  dirname + "/Ut4sq_{:.6f}/".format(Ut4sq)
                    outfilename = outdir + prefix
                    
#                     loose_fit = loose[i_ut4sq, i_dm2*len(Um4sq_list)+i_um4sq, 3]
                    
#                     if loose_fit == -np.inf:
#                         make_fit_dag(f, str(i), fluxdir, point, outdir, osg)
#                         print_vars(f, str(i), 'prefix', prefix.replace('.json', ''))
#                         i+=1
#                     else:
#                         continue
#                     dev = -(loose_fit - loose_null[3]) * np.log10(np.exp(1.))
#                     if dev < 0.:
#                     if dev > -1.:
#                     if dev > 0 and dev < 3.:
#                     if dev > 3 and dev < 20.:
#                     if dev > 20:
                    if True:
                        if not doesfilehaveBayes(outfilename, False):

                    
#                             print(dev)
                            print(Um4sq_list[i_um4sq], Ut4sq_list[i_ut4sq])
                            print(utils.Usq_to_thetas(0.,Um4sq = Um4sq_list[i_um4sq], Ut4sq=Ut4sq_list[i_ut4sq]))
                            make_fit_dag(f, str(i), fluxdir, point, outdir, osg)
                            print_vars(f, str(i), 'prefix', prefix.replace('.json', ''))
#                             f.write('Retry ' + str(i) + " 10\n")


                            i+=1



In [ ]:
import subprocess
def get_err(filename, delete=False):
    with open(filename) as f:
#         print(filename)
        content = f.readlines()
        line = content[1]
        err = float(line.split()[-1])
#     err=float(subprocess.check_output("awk 'NR==2' {} | cut   -b 85-".format(filename), shell=True, text=True))
    
    if err>0.05:
        print(filename, "has err {}".format(err))
        if delete:
            print("Deleting")
            filebase = filename.replace('.stats.dat', '')
            filenames = glob.glob(filebase+"*")

            for filetodelete in filenames:
                os.remove(filetodelete)
                
    return err

In [ ]:
%%time
dirname = "/data/user/adiaz/Bayes_Run2/"
Ue4sq= 0.
del14=0.
errs = []
for i_ut4sq, Ut4sq in enumerate(Ut4sq_list):

    print(Ut4sq)
    if Ut4sq == 0.:
    

        prefix =  "{:.6f}_{:.6f}_{:.6f}.json".format(0.,0.,0.,)
        point = utils.get_point_string()
        fluxdir = "/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/Fluxes/AIRS_mceq121_2011_sib_HG_E3/Ut4sq_" + str(format(float(Ut4sq), '.6f')) + '/'

        outdir = dirname + "/Ut4sq_{:.6f}/".format(Ut4sq)
        outfilename = outdir + prefix

        if doesfilehaveBayes(outfilename):

            err = get_err(outfilename.replace("json", "stats.dat"))
            errs.append(err)

    for i_um4sq, Um4sq in enumerate(Um4sq_list):
#                 print(Um4sq)
        for i_dm2, dm2 in enumerate(dm2_list):
#                     print(dm2)
            for del24 in d24_list:

                th14, th24, th34 = utils.Usq_to_thetas(Ue4sq=Ue4sq, Um4sq=Um4sq, Ut4sq=Ut4sq)
                point = utils.get_point_string(dm2 = dm2, th14 = th14, th24 = th24, th34 = th34, d24 = np.pi)
                prefix =  "{:.6f}_{:.6f}_{:.6f}.json".format(dm2, th24, th34)
#                         print(prefix)

                fluxdir = "/data/ana/SterileNeutrino/IC86/HighEnergy/Analysis/th34/Fluxes/AIRS_mceq121_2011_sib_HG_E3/Ut4sq_" + str(format(float(Ut4sq), '.6f')) + '/'
                outdir =  dirname + "/Ut4sq_{:.6f}/".format(Ut4sq)
                outfilename = outdir + prefix

                loose_fit = loose[i_ut4sq, i_dm2*len(Um4sq_list)+i_um4sq, 3]


                dev = -(loose_fit - loose_null[3]) * np.log10(np.exp(1.))
                if True:
#                 if dev > -1.:
#                 if dev < 0.:
#                 if dev > 0 and dev < 3.:

#                 if dev:
#                     if doesfilehaveBayes(outfilename, False):
                    if exists(outfilename.replace("json", "stats.dat")):
                        err = get_err(outfilename.replace("json", "stats.dat"), delete=False)
                        errs.append(err)


#                             f.write('Retry ' + str(i) + " 10\n")
errs = np.array(errs)


In [ ]:
import matplotlib.pyplot as plt
bins = [0.,.05,1.]
plt.hist(errs, bins = bins)

In [ ]:
import matplotlib.pyplot as plt
bins = np.linspace(0, 1, 101)
plt.hist(errs, bins = bins)

In [ ]:
(errs>0.05).sum()/len(errs)